# Phishing URL Detection - Initial Exploration

## Business Context

**Stakeholder**: Small security consulting firm (internal IT security team)

**Problem Statement**: Staff members are increasingly targeted by phishing attacks via suspicious URLs in emails and messages. The firm needs a quick-check tool where any employee can paste a URL and get an instant risk assessment.

**Business Goal**: 
- Reduce phishing click-through incidents by 70% within 6 months
- Provide real-time URL risk scoring with explainable features
- Enable rapid security awareness training by showing WHY a URL is risky (excessive subdomains, suspicious characters, unusual length, etc.)

**Success Criteria**: A lightweight classification model that can flag risky URLs with high precision, minimizing false alarms while catching genuine threats.

## CRISP-DM Methodology

We're following the **CRISP-DM** (Cross-Industry Standard Process for Data Mining) framework:

1. **Business Understanding** - Define the phishing detection problem and success metrics
2. **Data Understanding** (this notebook) - Load, explore, and identify key patterns in URL features
3. **Data Preparation** - Feature engineering, cleaning, train/test splits
4. **Modeling** - Build and tune classification models
5. **Evaluation** - Assess performance against business KPIs
6. **Deployment** - Create a simple prediction interface

**This session**: Phases 1-2 (Business Understanding + Data Understanding)

## 1. Environment Setup

In [ ]:
# Unpack the dataset archive
import zipfile
import os

# Extract datasets from archive
with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
    zip_ref.extractall('data/')
    
print("Extracted files:")
for file in os.listdir('data/'):
    size_mb = os.path.getsize(f'data/{file}') / (1024 * 1024)
    print(f"  - {file} ({size_mb:.1f} MB)")

In [ ]:
# Load necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully")

In [ ]:
# Load the first dataset to get a feel for the data structure
df = pd.read_csv('data/dataset1.csv')

print(f"Dataset shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nColumn names:")
print(df.columns.tolist())

## 2. Data Loading and Initial Inspection

In [ ]:
# Check all datasets - test loading with proper error handling
import glob
import warnings

dataset_files = sorted(glob.glob('data/dataset*.csv'))

print("Comparing all datasets:\n")
for file in dataset_files:
    issues = []
    
    try:
        # Try UTF-8 first
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            df_temp = pd.read_csv(file, encoding='utf-8', low_memory=False)
            if w:
                for warning in w:
                    if 'DtypeWarning' in str(warning.category):
                        issues.append("mixed dtypes")
    except UnicodeDecodeError:
        # Try latin-1 encoding
        try:
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                df_temp = pd.read_csv(file, encoding='latin-1', on_bad_lines='skip', low_memory=False)
                issues.append("encoding: latin-1")
        except Exception as e:
            print(f"{file}")
            print(f"  FAILED: {str(e)[:80]}")
            print()
            continue
    except Exception as e:
        # Handle parsing errors
        try:
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                df_temp = pd.read_csv(file, encoding='latin-1', on_bad_lines='skip', low_memory=False)
                issues.append("parsing errors")
        except Exception as e2:
            print(f"{file}")
            print(f"  FAILED: {str(e2)[:80]}")
            print()
            continue
    
    print(f"{file}")
    print(f"  Shape: {df_temp.shape[0]:,} rows × {df_temp.shape[1]} columns")
    print(f"  Columns: {df_temp.columns.tolist()[:5]}...")
    if issues:
        print(f"  Issues: {', '.join(issues)}")
    print()

## 3. Dataset Comparison and Analysis

## Key Finding: Heterogeneous Dataset Collection

**Discovery**: While the Kaggle page mentions "diverse approaches," loading the data reveals just how different these datasets are. This is the first time we're seeing the actual structure directly.

**What We Found**:
- 6 datasets with wildly different structures (32 to 112 columns)
- Sizes range from 10k to 235k rows
- Different feature engineering philosophies (some extract 100+ features, others focus on 14 core signals)
- dataset5 has data quality issues (parsing errors, mixed data types)

**Implication for Our Business Case**:

Our stakeholder needs **explainability** - when a URL is flagged, staff need to understand WHY. This means we need human-readable feature names, not just numerical indicators.

**Selection Criteria**:
1. Contains raw URL or domain column (for showing examples)
2. Feature names are interpretable (e.g., "NumDots" not "feature_47")
3. Sufficient training data (10k+ minimum)
4. Clean loading (no major data quality issues)

**Top Candidates**:
- **dataset3**: 11k rows, 89 features - has 'url', 'length_url', 'nb_dots' (interpretable)
- **dataset4**: 235k rows, 56 features - has 'URL', 'Domain', 'URLLength' (LARGEST, interpretable)
- **dataset6**: 10k rows, 50 features - has 'NumDots', 'SubdomainLevel' (interpretable but smallest)

**Decision**: Examine dataset3 and dataset4 in detail. dataset4's size (235k) is compelling for model performance, but we need to verify feature interpretability first.

## 4. Dataset4 Detailed Examination

In [ ]:
# Examine dataset4 in detail (largest candidate at 235k rows)
df4 = pd.read_csv('data/dataset4.csv')

print(f"=== Dataset4 Overview ===")
print(f"Shape: {df4.shape[0]:,} rows × {df4.shape[1]} columns\n")

# Create a DataFrame to display column names in a clean format
col_info = pd.DataFrame({
    'Column': df4.columns,
    'Type': df4.dtypes.astype(str).values
})

# Display first 15 and last 5 features
print("Features (showing first 15 and last 5):")
display(pd.concat([col_info.head(15), 
                  pd.DataFrame({'Column': ['...'], 'Type': ['...']}),
                  col_info.tail(5)])
       .style.set_caption("Dataset4 Feature Overview"))

In [ ]:
# Sample rows with key features to understand what the data captures
# Select a diverse subset of features that tell different parts of the story

key_features = [
    'URL',                    # What we're analyzing
    'Domain',                 # Domain extracted
    'URLLength',              # Structure: length
    'NoOfSubDomain',          # Structure: subdomain count
    'IsHTTPS',                # Security: HTTPS flag
    'IsDomainIP',             # Security: IP address instead of domain
    'Bank',                   # Content: banking keyword
    'Pay',                    # Content: payment keyword  
    'Crypto',                 # Content: crypto keyword
    'HasPasswordField',       # Behavior: has password input
    'HasObfuscation',         # Behavior: obfuscated characters
    'label'                   # Ground truth (0=legit, 1=phishing)
]

# Show 10 samples to get diverse examples
sample_df = df4[key_features].head(10)

# Set pandas display options for better readability
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', None)

print("=== Sample URLs with Key Features ===\n")
print(sample_df.to_string(index=False))

# Reset display options
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')

In [ ]:
# Verify what the label values actually mean
# Check class distribution first
print("=== Label Distribution ===")
print(df4['label'].value_counts().sort_index())
print(f"\nClass balance:")
print(df4['label'].value_counts(normalize=True).sort_index())

print("\n=== Examples of label=0 ===")
label_0_sample = df4[df4['label'] == 0][['URL', 'Domain', 'IsHTTPS', 'HasObfuscation', 'Bank', 'label']].head(5)
print(label_0_sample.to_string(index=False))

print("\n=== Examples of label=1 ===")
label_1_sample = df4[df4['label'] == 1][['URL', 'Domain', 'IsHTTPS', 'HasObfuscation', 'Bank', 'label']].head(5)
print(label_1_sample.to_string(index=False))

## Correction: Label Encoding

**Previous Assumption (INCORRECT)**: In cell-8, we assumed `label=0` meant legitimate and `label=1` meant phishing.

**Evidence from verification**:
- **label=0** examples: `f0519141.xsph.ru`, `shprakserf.gq`, `kuradox92.lima-city.de` - suspicious domains with random strings, uncommon TLDs
- **label=1** examples: `uni-mainz.de`, `voicefmradio.co.uk`, `rewildingargentina.org` - recognizable legitimate organizations

**Actual encoding**:
- **0 = phishing** (42.8% of dataset, 100,945 samples)
- **1 = legitimate** (57.2% of dataset, 134,850 samples)

**Lesson**: Always verify assumptions about data encoding before proceeding with analysis. The comment in cell-8 remains as a reminder of this error.

## 5. Data Quality Assessment

In [ ]:
# Data quality check for dataset4
print("=== Dataset4 Data Quality Check ===\n")

# Check for missing values
print("Missing values per column:")
missing = df4.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) > 0:
    print(missing_cols)
else:
    print("No missing values in any column")

print(f"\nTotal missing values: {df4.isnull().sum().sum()}")
print(f"Percentage of data missing: {(df4.isnull().sum().sum() / (df4.shape[0] * df4.shape[1]) * 100):.2f}%")

# Check for duplicates
print(f"\n=== Duplicate Check ===")
print(f"Duplicate rows: {df4.duplicated().sum()}")
print(f"Duplicate URLs: {df4['URL'].duplicated().sum()}")

# Basic data types check
print(f"\n=== Data Types ===")
print(f"Numeric columns: {len(df4.select_dtypes(include=['int64', 'float64']).columns)}")
print(f"Text columns: {len(df4.select_dtypes(include=['object']).columns)}")

## 6. Final Dataset Selection

## Dataset Selection Decision

Based on initial inspection, we focused on **dataset4** due to its size (235k samples - 2.5x larger than the next biggest). After examination, we confirm it meets our requirements:

**Why dataset4:**
1. **Size advantage**: 235,795 samples provide robust training data  
2. **Data completeness**: Zero missing values across all 56 columns
3. **Interpretable features**: Clear names that can be explained to users
4. **Contains raw URL and domain**: Essential for showing which URL was flagged
5. **Verified encoding**: Confirmed label meanings (0=phishing, 1=legitimate)

**Why NOT combine datasets:**
- Each dataset has completely different feature extraction approaches (32 to 112 columns)
- No standardized feature definitions across datasets
- Dataset4 alone is sufficient for our needs

**Note**: While dataset3 and dataset6 were identified as candidates, we proceeded with dataset4 after confirming it had all necessary characteristics. Further comparison was deemed unnecessary given dataset4's clear advantages.

# Dataset4 Feature Deep Dive

Now that we've selected dataset4, we need to understand what each feature actually measures. This understanding is critical for:

1. **Rule-based system** - Identifying clear red flags for instant detection
2. **ML model** - Understanding feature importance and model decisions  
3. **Explainability** - Explaining to users WHY a URL was flagged

## Feature Investigation: Obfuscation Metrics

Three related features measure obfuscation in URLs:
- `HasObfuscation` (binary flag)
- `NoOfObfuscatedChar` (count)
- `ObfuscationRatio` (proportion)

In [ ]:
# Explore obfuscation features with better visualization
import matplotlib.pyplot as plt

# Create a cleaner crosstab display
print("=== Obfuscation Distribution by Label ===\n")
obf_crosstab = pd.crosstab(df4['HasObfuscation'], df4['label'], 
                           margins=True, margins_name="Total")
obf_crosstab.columns = ['Phishing (0)', 'Legitimate (1)', 'Total']
obf_crosstab.index = ['No Obfuscation', 'Has Obfuscation', 'Total']

# Display as a styled dataframe
display(obf_crosstab.style.set_caption("URL Obfuscation by Label")
        .format('{:,}')
        .set_table_styles([{'selector': 'caption', 
                           'props': [('font-size', '16px'), ('font-weight', 'bold')]}]))

# Show percentages
print("\n=== Percentage Distribution ===")
obf_pct = pd.crosstab(df4['HasObfuscation'], df4['label'], normalize='columns') * 100
obf_pct.columns = ['Phishing (0)', 'Legitimate (1)']
obf_pct.index = ['No Obfuscation %', 'Has Obfuscation %']
display(obf_pct.style.format('{:.2f}%')
        .background_gradient(cmap='YlOrRd', axis=None)
        .set_caption("Percentage of URLs with Obfuscation by Label"))

# Statistics summary in a cleaner format
print("\n=== Obfuscation Character Statistics ===")
stats = df4.groupby('label')['NoOfObfuscatedChar'].describe()[['mean', 'std', 'max']]
stats.index = ['Phishing (0)', 'Legitimate (1)']
display(stats.style.format('{:.4f}')
        .set_caption("Number of Obfuscated Characters Statistics"))

In [ ]:
# Show sample URLs with and without obfuscation
print("=== Sample URLs WITH Obfuscation (Phishing only) ===")
obf_samples = df4[df4['HasObfuscation'] == 1][['URL', 'NoOfObfuscatedChar', 'ObfuscationRatio']].head(5)
obf_samples['URL'] = obf_samples['URL'].str[:80] + '...'  # Truncate long URLs for display
display(obf_samples.style.set_caption("URLs with Obfuscation (all are phishing)")
        .format({'ObfuscationRatio': '{:.2%}'}))

print("\n=== Sample Legitimate URLs (Never have obfuscation) ===")
legit_samples = df4[df4['label'] == 1][['URL', 'Domain', 'IsHTTPS']].head(5)
display(legit_samples.style.set_caption("Sample Legitimate URLs"))

### Obfuscation Findings

**Key Discovery**: Obfuscation is a **perfect indicator** of phishing in this dataset:
- **100% of legitimate URLs** have NO obfuscation
- Only **0.48% of phishing URLs** use obfuscation (485 out of 100,945)
- When present, obfuscation involves URL percent-encoding (e.g., `%20` for space, `%23` for #)

**Examples of obfuscation patterns found**:
- `banco%20davivienda` - encoding spaces in bank names
- `%26%29%24%21` - encoding special characters
- `%5bemail%5d` - encoding brackets around email placeholders

**Implication for rule-based system**: 
- `if HasObfuscation == 1: classify as PHISHING` (100% precision on this dataset)
- However, this only catches 0.48% of phishing URLs, so we need additional features

## Feature Investigation: IsDomainIP

This binary feature indicates whether the URL uses an IP address instead of a domain name.
Example: `http://192.168.1.1/login` vs `http://google.com/login`

In [ ]:
# Investigate IsDomainIP feature
print("=== IsDomainIP Distribution by Label ===\n")

# Create crosstab
ip_crosstab = pd.crosstab(df4['IsDomainIP'], df4['label'], 
                          margins=True, margins_name="Total")
ip_crosstab.columns = ['Phishing (0)', 'Legitimate (1)', 'Total']
ip_crosstab.index = ['Domain Name', 'IP Address', 'Total']

# Display counts
display(ip_crosstab.style.set_caption("URLs using IP vs Domain Name")
        .format('{:,}'))

# Show percentages
print("\n=== Percentage Distribution ===")
ip_pct = pd.crosstab(df4['IsDomainIP'], df4['label'], normalize='columns') * 100
ip_pct.columns = ['Phishing (0)', 'Legitimate (1)']
ip_pct.index = ['Domain Name %', 'IP Address %']
display(ip_pct.style.format('{:.2f}%')
        .background_gradient(cmap='RdYlGn_r', axis=None)
        .set_caption("Percentage of URLs using IP addresses"))

# Sample URLs with IP addresses
print("\n=== Sample URLs using IP addresses ===")
ip_samples = df4[df4['IsDomainIP'] == 1][['URL', 'Domain', 'label']].head(10)
ip_samples['URL'] = ip_samples['URL'].str[:60] + '...'
ip_samples['label'] = ip_samples['label'].map({0: 'Phishing', 1: 'Legitimate'})
display(ip_samples.style.set_caption("Examples of URLs using IP addresses"))

### IsDomainIP Findings

**Initial assumption**: Based on the feature name, I assume this indicates URLs using IP addresses instead of domain names.

**Verification from data**: Looking at sample URLs confirms this assumption - URLs with IsDomainIP=1 contain IP addresses in the Domain field.

**Key Discovery**: IP-based URLs are **extremely rare** and **only appear in phishing**:
- **100% of legitimate URLs** use domain names (0% use IP addresses)  
- Only **0.06% of phishing URLs** use IP addresses (58 out of 100,945)
- All 58 URLs with IP addresses are labeled as phishing

**Why this matters for phishing detection**:
- Legitimate websites use memorable domain names for branding and trust
- Phishers sometimes use IP addresses to avoid domain registration or to evade domain-based blocklists

**Implication for rule-based system**:
- `if IsDomainIP == 1: classify as PHISHING` (100% precision on this dataset)
- Like obfuscation, this is a perfect indicator but catches very few phishing URLs (0.06%)

## Feature Investigation: URLLength

I assume longer URLs might be associated with phishing (hiding the real domain with long paths/parameters).

In [ ]:
# Investigate URLLength distribution
import matplotlib.pyplot as plt

# Basic statistics
print("=== URLLength Statistics by Label ===\n")
url_stats = df4.groupby('label')['URLLength'].describe()
url_stats.index = ['Phishing (0)', 'Legitimate (1)']
display(url_stats.style.format('{:.1f}')
        .set_caption("URL Length Statistics"))

# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist([df4[df4['label']==0]['URLLength'], 
              df4[df4['label']==1]['URLLength']], 
             bins=50, alpha=0.7, label=['Phishing', 'Legitimate'], color=['red', 'green'])
axes[0].set_xlabel('URL Length')
axes[0].set_ylabel('Count')
axes[0].set_title('URL Length Distribution by Label')
axes[0].legend()
axes[0].set_xlim(0, 200)  # Focus on main range

# Box plot - using tick_labels instead of labels for compatibility
box_data = [df4[df4['label']==0]['URLLength'], df4[df4['label']==1]['URLLength']]
bp = axes[1].boxplot(box_data, tick_labels=['Phishing', 'Legitimate'], patch_artist=True)
bp['boxes'][0].set_facecolor('salmon')
bp['boxes'][1].set_facecolor('lightgreen')
axes[1].set_ylabel('URL Length')
axes[1].set_title('URL Length Comparison')
axes[1].set_ylim(0, 200)  # Focus on main range

plt.tight_layout()
plt.show()

# Check extremes
print("\n=== Extreme URL Lengths ===")
print(f"Shortest phishing URL: {df4[df4['label']==0]['URLLength'].min()} chars")
print(f"Longest phishing URL: {df4[df4['label']==0]['URLLength'].max()} chars")
print(f"Shortest legitimate URL: {df4[df4['label']==1]['URLLength'].min()} chars")
print(f"Longest legitimate URL: {df4[df4['label']==1]['URLLength'].max()} chars")

In [ ]:
# Look at sample URLs by length categories
print("=== Sample URLs by Length Category ===\n")

# Very short URLs (< 20 chars)
print("VERY SHORT URLs (<20 chars):")
very_short = df4[df4['URLLength'] < 20][['URL', 'URLLength', 'label']].head(3)
very_short['label'] = very_short['label'].map({0: 'Phishing', 1: 'Legitimate'})
display(very_short)

# Moderate length URLs (30-50 chars) 
print("\nMODERATE LENGTH URLs (30-50 chars):")
moderate = df4[(df4['URLLength'] >= 30) & (df4['URLLength'] <= 50)][['URL', 'URLLength', 'label']].head(3)
moderate['label'] = moderate['label'].map({0: 'Phishing', 1: 'Legitimate'})
display(moderate)

# Very long URLs (>150 chars)
print("\nVERY LONG URLs (>150 chars):")
very_long = df4[df4['URLLength'] > 150][['URL', 'URLLength', 'label']].head(3)
very_long['URL'] = very_long['URL'].str[:80] + '...'  # Truncate for display
very_long['label'] = very_long['label'].map({0: 'Phishing', 1: 'Legitimate'})
display(very_long)

In [ ]:
# Verify exact URLLength statistics
phishing_urls = df4[df4['label'] == 0]['URLLength']
legit_urls = df4[df4['label'] == 1]['URLLength']

print("=== Exact URLLength Statistics ===")
print(f"\nPhishing URLs (label=0, n={len(phishing_urls)}):")
print(f"  Mean: {phishing_urls.mean():.1f} chars")
print(f"  Median: {phishing_urls.median():.1f} chars")
print(f"  Min: {phishing_urls.min()} chars")
print(f"  Max: {phishing_urls.max()} chars")

print(f"\nLegitimate URLs (label=1, n={len(legit_urls)}):")
print(f"  Mean: {legit_urls.mean():.1f} chars")  
print(f"  Median: {legit_urls.median():.1f} chars")
print(f"  Min: {legit_urls.min()} chars")
print(f"  Max: {legit_urls.max()} chars")

print(f"\nComparison:")
print(f"  Legitimate URLs are {legit_urls.mean() - phishing_urls.mean():.1f} chars longer on average")
print(f"  Legitimate median is {legit_urls.median() - phishing_urls.median():.1f} chars longer")

### URLLength Findings

**Initial assumption**: I assumed longer URLs would be associated with phishing (to hide domains with complex paths).

**Actual findings from data** (verified in cell above):
- **Phishing URLs**: Mean 45.7 chars, Median 34.0 chars
- **Legitimate URLs**: Mean 26.2 chars, Median 26.0 chars  
- **My assumption was partially correct**: Phishing URLs ARE longer on average!

**Distribution insights**:
- Phishing URLs are 19.5 chars longer on average
- Phishing median is 8 chars longer than legitimate
- Phishing has extreme outliers (max: 6097 chars vs legitimate max: 57 chars)
- Legitimate URLs are very consistent (15-57 chars range)

**Why phishing URLs tend to be longer**:
- Complex redirects and tracking parameters
- Attempts to hide the real destination with long paths
- Subdomain spoofing (e.g., `secure.bank.com.evil.com/...`)
- URL shorteners that expand to long URLs

**Why legitimate URLs stay short**:
- Good UX practices favor short, memorable URLs
- SEO best practices recommend concise URLs
- Professional sites use proper domain names without complex tricks

**Implication for rule-based system**:
- URLs over 57 chars are ALWAYS phishing in this dataset (100% precision)
- Could use threshold: `if URLLength > 57: classify as PHISHING`
- This would catch many phishing URLs with perfect precision

## Feature Investigation: IsHTTPS

I assume legitimate sites are more likely to use HTTPS for security and trust signals.

In [ ]:
# Investigate IsHTTPS feature
print("=== HTTPS Usage by Label ===\n")

# Create crosstab
https_crosstab = pd.crosstab(df4['IsHTTPS'], df4['label'], 
                             margins=True, margins_name="Total")
https_crosstab.columns = ['Phishing (0)', 'Legitimate (1)', 'Total']
https_crosstab.index = ['HTTP', 'HTTPS', 'Total']

# Display counts
display(https_crosstab.style.set_caption("HTTP vs HTTPS Usage")
        .format('{:,}')
        .set_table_styles([{'selector': 'caption',
                           'props': [('font-size', '14px'), ('font-weight', 'bold')]}]))

# Show percentages
print("\n=== Percentage Distribution ===")
https_pct = pd.crosstab(df4['IsHTTPS'], df4['label'], normalize='columns') * 100
https_pct.columns = ['Phishing (%)', 'Legitimate (%)']
https_pct.index = ['HTTP', 'HTTPS']
display(https_pct.style.format('{:.1f}%')
        .background_gradient(cmap='RdYlGn', axis=1)
        .set_caption("Percentage of URLs using HTTPS"))

# Visualization
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 6))

# Stacked bar chart
labels = ['Phishing', 'Legitimate']
http_vals = [https_pct.iloc[0, 0], https_pct.iloc[0, 1]]
https_vals = [https_pct.iloc[1, 0], https_pct.iloc[1, 1]]

x = range(len(labels))
width = 0.5

p1 = ax.bar(x, http_vals, width, label='HTTP', color='#ff9999')
p2 = ax.bar(x, https_vals, width, bottom=http_vals, label='HTTPS', color='#90ee90')

ax.set_ylabel('Percentage (%)')
ax.set_title('HTTP vs HTTPS Usage by Label')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# Add percentage labels on bars
for i, (http_val, https_val) in enumerate(zip(http_vals, https_vals)):
    ax.text(i, http_val/2, f'{http_val:.1f}%', ha='center', va='center')
    ax.text(i, http_val + https_val/2, f'{https_val:.1f}%', ha='center', va='center')

plt.tight_layout()
plt.show()

# Sample URLs
print("\n=== Sample HTTP Phishing URLs ===")
http_phishing = df4[(df4['IsHTTPS'] == 0) & (df4['label'] == 0)][['URL', 'Domain']].head(3)
display(http_phishing)

print("\n=== Sample HTTPS Phishing URLs ===")
https_phishing = df4[(df4['IsHTTPS'] == 1) & (df4['label'] == 0)][['URL', 'Domain']].head(3)
display(https_phishing)

In [ ]:
# Calculate exact percentages for verification
phishing_df = df4[df4['label'] == 0]
legit_df = df4[df4['label'] == 1]

print("=== Exact HTTPS Statistics ===")
print(f"\nPhishing URLs (label=0, n={len(phishing_df)}):")
print(f"  HTTP (IsHTTPS=0): {len(phishing_df[phishing_df['IsHTTPS']==0]):,} ({len(phishing_df[phishing_df['IsHTTPS']==0])/len(phishing_df)*100:.2f}%)")
print(f"  HTTPS (IsHTTPS=1): {len(phishing_df[phishing_df['IsHTTPS']==1]):,} ({len(phishing_df[phishing_df['IsHTTPS']==1])/len(phishing_df)*100:.2f}%)")

print(f"\nLegitimate URLs (label=1, n={len(legit_df)}):")
print(f"  HTTP (IsHTTPS=0): {len(legit_df[legit_df['IsHTTPS']==0]):,} ({len(legit_df[legit_df['IsHTTPS']==0])/len(legit_df)*100:.2f}%)")
print(f"  HTTPS (IsHTTPS=1): {len(legit_df[legit_df['IsHTTPS']==1]):,} ({len(legit_df[legit_df['IsHTTPS']==1])/len(legit_df)*100:.2f}%)")

### IsHTTPS Findings

**Initial assumption**: I assumed legitimate sites would use HTTPS more for security and trust.

**Actual findings from data** (verified from cell above):
- **Phishing**: 50.78% HTTP, 49.22% HTTPS (almost evenly split)
- **Legitimate**: 0% HTTP, 100% HTTPS (ALL legitimate URLs use HTTPS)
- **Critical discovery**: ALL legitimate URLs in this dataset use HTTPS!

**Key insights**:
- **HTTPS is mandatory for legitimate sites** in this dataset (100% usage)
- **HTTP is a strong phishing indicator**: If HTTP, then 100% chance it's phishing in this dataset
- Almost half of phishing sites (49%) have adopted HTTPS to appear trustworthy

**Why this pattern exists**:
- Modern browsers mark HTTP sites as "Not Secure"
- Legitimate businesses migrated to HTTPS for SEO and trust
- Free SSL certificates (Let's Encrypt) made HTTPS accessible to phishers too

**Implication for rule-based system**:
- `if IsHTTPS == 0: classify as PHISHING` (100% precision in this dataset!)
- This catches 50.78% of phishing URLs (much better than obfuscation or IP features)
- Combined with other perfect indicators, we're building a strong rule set